# ErgTrainer raw BLE signal analysis

This notebook preserves each raw packet while decoding the standard BLE layouts for heart rate, FTMS Indoor Bike Data, and Cycling Power. Use a controlled ride (for example, deliberate cadence and power changes) plus a Garmin, Strava, or MyWhoosh export to validate any mapping.

The application writes `timestamp_utc`, `source`, `device`, `delivery_method`, `service_uuid`, `characteristic_uuid`, and `raw_hex`. Notifications and polling are both retained; start by filtering to `notification` when comparing packet streams.

In [ ]:
from pathlib import Path
import os
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

# Leave this as None to use the newest raw-signals CSV created by ErgTrainer.
# Example: RECORDING_FILE = Path(r"C:\Users\you\Documents\ErgTrainer\Recordings\raw-signals-20260929-120000000.csv")
RECORDING_FILE = None
RECORDINGS_DIR = Path.home() / "Documents" / "ErgTrainer" / "Recordings"

def select_recording(path=RECORDING_FILE, recordings_dir=RECORDINGS_DIR):
    if path is not None:
        path = Path(path)
        if not path.is_file():
            raise FileNotFoundError(f"Recording not found: {path}")
        return path

    candidates = sorted(recordings_dir.glob("raw-signals-*.csv"), key=lambda item: item.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError(
            f"No raw-signals CSV found in {recordings_dir}. Record a ride in ErgTrainer or set RECORDING_FILE."
        )
    return candidates[-1]


In [ ]:
REQUIRED_COLUMNS = {
    "timestamp_utc", "source", "device", "delivery_method",
    "service_uuid", "characteristic_uuid", "raw_hex",
}

def load_raw_recording(path):
    frame = pd.read_csv(path)
    missing = REQUIRED_COLUMNS.difference(frame.columns)
    if missing:
        raise ValueError(f"Not an ErgTrainer raw-signal CSV; missing columns: {sorted(missing)}")

    frame["timestamp_utc"] = pd.to_datetime(frame["timestamp_utc"], utc=True, errors="coerce")
    frame["payload"] = frame["raw_hex"].fillna("").map(lambda value: bytes.fromhex(value))
    frame["characteristic_uuid"] = frame["characteristic_uuid"].str.lower()
    return frame.sort_values("timestamp_utc").reset_index(drop=True)

recording_path = select_recording()
raw = load_raw_recording(recording_path)
print(f"Loaded {len(raw):,} packets from {recording_path}")
display(raw[["timestamp_utc", "source", "device", "delivery_method", "characteristic_uuid", "raw_hex"]].head())
display(raw.groupby(["source", "characteristic_uuid", "delivery_method"]).size().rename("packets").reset_index())


In [ ]:
# Inspect the bytes before accepting any decoder result. `word_le_XX` starts at byte XX.
def add_raw_views(frame):
    result = frame.copy()
    max_length = max(result["payload"].map(len), default=0)
    for index in range(max_length):
        result[f"byte_{index:02d}"] = result["payload"].map(lambda value: value[index] if len(value) > index else pd.NA)
    for index in range(max_length - 1):
        result[f"word_le_{index:02d}"] = result["payload"].map(
            lambda value: value[index] | (value[index + 1] << 8) if len(value) > index + 1 else pd.NA
        )
    return result

raw_view = add_raw_views(raw)
display(raw_view.filter(regex=r"^(timestamp_utc|source|characteristic_uuid|raw_hex|byte_|word_le_)").head())


In [ ]:
HR_MEASUREMENT_UUID = "00002a37-0000-1000-8000-00805f9b34fb"
FTMS_INDOOR_BIKE_DATA_UUID = "00002ad2-0000-1000-8000-00805f9b34fb"
CYCLING_POWER_MEASUREMENT_UUID = "00002a63-0000-1000-8000-00805f9b34fb"

def u16(data, offset):
    return int.from_bytes(data[offset:offset + 2], "little", signed=False)

def s16(data, offset):
    return int.from_bytes(data[offset:offset + 2], "little", signed=True)

def u24(data, offset):
    return int.from_bytes(data[offset:offset + 3], "little", signed=False)

def decode_heart_rate(data):
    if len(data) < 2:
        return {}
    flags = data[0]
    is_uint16 = flags & 0x01
    width = 2 if is_uint16 else 1
    if len(data) < 1 + width:
        return {}
    result = {"heart_rate_bpm": u16(data, 1) if is_uint16 else data[1]}
    offset = 1 + width
    if flags & 0x08 and len(data) >= offset + 2:  # Energy Expended present
        result["energy_expended_kj"] = u16(data, offset)
        offset += 2
    if flags & 0x10 and len(data) >= offset + 2:  # RR intervals, 1/1024 seconds
        rr_seconds = [u16(data, index) / 1024 for index in range(offset, len(data) - 1, 2)]
        result["rr_interval_seconds"] = rr_seconds
        result["rr_interval_mean_seconds"] = sum(rr_seconds) / len(rr_seconds)
    return result

def decode_ftms_indoor_bike(data):
    if not data:
        return {}
    flags, offset, result = data[0], 1, {}
    # FTMS bit 0 means More Data: when set, instantaneous speed is omitted.
    if not flags & 0x01 and len(data) >= offset + 2:
        result["speed_kph"] = u16(data, offset) / 100
        offset += 2
    if flags & 0x02 and len(data) >= offset + 2:
        result["average_speed_kph"] = u16(data, offset) / 100
        offset += 2
    if flags & 0x04 and len(data) >= offset + 2:
        result["cadence_rpm"] = u16(data, offset) / 2
        offset += 2
    if flags & 0x08 and len(data) >= offset + 2:
        result["average_cadence_rpm"] = u16(data, offset) / 2
        offset += 2
    if flags & 0x10 and len(data) >= offset + 3:
        result["total_distance_m"] = u24(data, offset)
        offset += 3
    if flags & 0x20 and len(data) >= offset + 2:
        result["resistance_level"] = s16(data, offset)
        offset += 2
    if flags & 0x40 and len(data) >= offset + 2:
        result["power_watts"] = s16(data, offset)
        offset += 2
    if flags & 0x80 and len(data) >= offset + 2:
        result["average_power_watts"] = s16(data, offset)
    return result


In [ ]:
def decode_cycling_power(data):
    if len(data) < 4:
        return {}
    flags, offset = u16(data, 0), 2
    result = {"power_watts": s16(data, offset)}  # Instantaneous Power is always present.
    offset += 2
    if flags & 0x01 and len(data) >= offset + 1:
        result["pedal_power_balance_percent"] = data[offset] / 2
        offset += 1
    if flags & 0x04 and len(data) >= offset + 2:
        offset += 2  # Accumulated Torque; retained in raw bytes for now.
    if flags & 0x10 and len(data) >= offset + 6:
        result["cumulative_wheel_revolutions"] = int.from_bytes(data[offset:offset + 4], "little")
        result["last_wheel_event_time_1024"] = u16(data, offset + 4)
        offset += 6
    if flags & 0x20 and len(data) >= offset + 4:
        result["cumulative_crank_revolutions"] = u16(data, offset)
        result["last_crank_event_time_1024"] = u16(data, offset + 2)
    return result

def decode_packet(row):
    characteristic = row["characteristic_uuid"]
    if characteristic == HR_MEASUREMENT_UUID:
        return decode_heart_rate(row["payload"])
    if characteristic == FTMS_INDOOR_BIKE_DATA_UUID:
        return decode_ftms_indoor_bike(row["payload"])
    if characteristic == CYCLING_POWER_MEASUREMENT_UUID:
        return decode_cycling_power(row["payload"])
    return {}

decoded = raw.join(pd.DataFrame(raw.apply(decode_packet, axis=1).tolist()))
display(decoded.dropna(axis=1, how="all").head(20))


In [ ]:
# Cycling Power cadence is derived from successive crank-event counters (which wrap at 16 bits).
def add_cycling_power_cadence(frame):
    result = frame.copy()
    required = {"cumulative_crank_revolutions", "last_crank_event_time_1024"}
    if not required.issubset(result.columns):
        return result
    is_cp = result["characteristic_uuid"].eq(CYCLING_POWER_MEASUREMENT_UUID)
    for _, indexes in result.loc[is_cp].groupby(["device", "characteristic_uuid"]).groups.items():
        series = result.loc[indexes].sort_values("timestamp_utc")
        revolutions = series["cumulative_crank_revolutions"]
        event_time = series["last_crank_event_time_1024"]
        revolution_delta = (revolutions - revolutions.shift()) % 65536
        time_delta = (event_time - event_time.shift()) % 65536
        cadence = revolution_delta * 60 * 1024 / time_delta
        result.loc[series.index, "cadence_rpm"] = cadence.where((time_delta > 0) & (cadence < 250))
    return result

decoded = add_cycling_power_cadence(decoded)
telemetry_columns = [column for column in ["power_watts", "cadence_rpm", "heart_rate_bpm", "speed_kph"] if column in decoded]
if telemetry_columns:
    display(decoded[["timestamp_utc", "source", "raw_hex", *telemetry_columns]].dropna(subset=telemetry_columns, how="all").head(30))
    decoded.set_index("timestamp_utc")[telemetry_columns].plot(subplots=True, figsize=(14, 8), grid=True, title="Decoded ErgTrainer telemetry")
    plt.tight_layout()
else:
    print("No standard heart-rate, FTMS, or Cycling Power packets were found. Inspect raw_view and the characteristic UUIDs.")


## Compare against Garmin, Strava, or MyWhoosh

Export a workout as FIT where available (Garmin and MyWhoosh commonly provide it), or as CSV/GPX (commonly available from Strava). The loaders below normalize timestamps and common telemetry names. Exports use different sampling and clock sources, so inspect the ride-start offset before comparing values.

In [ ]:
def load_fit_reference(path):
    from fitdecode import FitReader

    rows = []
    with FitReader(path) as fit:
        for frame in fit:
            if frame.frame_type == "data" and frame.name == "record":
                row = {field.name: field.value for field in frame.fields}
                rows.append({
                    "timestamp_utc": row.get("timestamp"),
                    "power_watts": row.get("power"),
                    "cadence_rpm": row.get("cadence"),
                    "heart_rate_bpm": row.get("heart_rate"),
                    "speed_kph": row.get("speed") * 3.6 if row.get("speed") is not None else None,
                })
    return normalise_reference(pd.DataFrame(rows))

def load_gpx_reference(path):
    root = ET.parse(path).getroot()
    namespace = {"gpx": "http://www.topografix.com/GPX/1/1"}
    rows = [{"timestamp_utc": point.findtext("gpx:time", namespaces=namespace)} for point in root.findall(".//gpx:trkpt", namespace)]
    return normalise_reference(pd.DataFrame(rows))

def normalise_reference(frame, timestamp_column="timestamp_utc"):
    result = frame.rename(columns={timestamp_column: "timestamp_utc"}).copy()
    result["timestamp_utc"] = pd.to_datetime(result["timestamp_utc"], utc=True, errors="coerce")
    return result.dropna(subset=["timestamp_utc"]).sort_values("timestamp_utc")

# For a CSV export, first inspect `pd.read_csv(path).columns`, then rename its time/data columns to the names above.
# reference = normalise_reference(pd.read_csv(r"path\to\strava-or-mywhoosh.csv").rename(columns={"time": "timestamp_utc", "watts": "power_watts"}))
# reference = load_fit_reference(r"path\to\garmin-or-mywhoosh.fit")
# reference = load_gpx_reference(r"path\to\strava.gpx")


In [ ]:
# After assigning `reference` in the prior cell, compare a metric.
# Align clocks first if needed, e.g. reference["timestamp_utc"] += pd.Timedelta(seconds=12).
# metric = "power_watts"  # also try cadence_rpm or heart_rate_bpm
# comparison = pd.concat([
#     decoded[["timestamp_utc", metric]].assign(origin="ErgTrainer raw BLE"),
#     reference[["timestamp_utc", metric]].assign(origin="reference export"),
# ]).dropna()
# comparison.pivot(index="timestamp_utc", columns="origin", values=metric).plot(figsize=(14, 4), grid=True, title=metric)
# plt.show()
